# Ask 7 — When Retrieval Fails, and the Fixes

Four engineered misses, four mechanisms, four fixes — each fix measured
against the full question set, because fixes can break hits. Fully
offline.

In [ ]:
# The pile: documents from the (fictional) Jefferson High School.
# Real enough to search, small enough to read whole.
PILE = {
 "handbook_academics": """S4.1 Grading scale. A 90-100, B 80-89, C 70-79, D 60-69.
Semester grades weight exams at 30 percent.
S4.2 Exam Retake Policy. This policy applies to final exams only. Students
receive one retake per semester, requested within ten school days. The
higher score stands.
S4.3 Grade appeals. Appeals go to the department head in writing within
fifteen school days of the posted grade.
S4.5 Late work. Assignments lose 10 percent per school day late, to a
maximum of 50 percent. Teachers may grant extensions for documented
emergencies.""",
 "handbook_schedule": """S2.0 Bell schedule. Regular days run eight periods,
8:15 AM to 3:20 PM.
S2.1 Wednesday schedule. Dismissal at 1:30 PM every Wednesday for staff
development.
S2.4 Late arrival. Students arriving after 8:30 AM sign in at the main
office with a note.""",
 "handbook_trips": """S5.1 Field trips require a signed permission form
submitted five school days in advance.
S5.2 Trip costs above 20 dollars qualify for the student activity fund.
S5.4 Chaperones must be approved district volunteers.""",
 "handbook_athletics": """S6.2 Eligibility. Athletes must hold a C average
during their season. Freshmen may try out for varsity teams.
S6.3 Petitions. A varsity roster spot for a freshman requires a coach's
petition to the athletic director.""",
 "robotics_minutes": """Robotics club meets Tuesdays in room 214. Regional
trip is April 18; bring your signed permission form by April 10. Dues are
15 dollars for the year.""",
 "clubs_list": """Active clubs: robotics (Tuesdays), debate (Thursdays),
art collective (Fridays), chess (lunch, library). Sign-up forms at the
student office.""",
 "bus_routes": """Routes 12 and 15 serve the north side. Final pickup at
4:45 PM outside door C. Activity buses run Tuesday and Thursday only.""",
 "cafeteria": """Lunch periods run 11:10, 11:55, and 12:40. Breakfast is
served from 7:40 AM. Menus post monthly on the food services page.""",
}
print(f"{len(PILE)} documents, {sum(len(t) for t in PILE.values())} characters total")

In [ ]:
def chunk_by_section(pile, overlap_sentences=1):
    """Cut on the S-section seams; carry a sentence of overlap across cuts."""
    chunks = []
    for doc, text in pile.items():
        parts, current, header = [], [], None
        for line in text.splitlines():
            if line.strip().startswith("S") and len(line) > 2 and line.strip()[1].isdigit():
                if current:
                    parts.append((header, " ".join(current)))
                header, current = line.strip().split()[0].rstrip("."), [line]
            else:
                current.append(line)
        if current:
            parts.append((header, " ".join(current)))
        for i, (header, body) in enumerate(parts):
            text_out = body
            if overlap_sentences and i > 0:
                prev_tail = parts[i-1][1].split(". ")[-1]
                text_out = prev_tail + " ... " + body
            chunks.append({"doc": doc, "section": header or doc, "text": " ".join(text_out.split())})
    return chunks

CHUNKS = chunk_by_section(PILE)
print(f"{len(CHUNKS)} chunks")
for c in CHUNKS[:3]:
    print(f"  [{c['doc']} {c['section']}] {c['text'][:70]}...")

In [ ]:
import math, re, collections

def words(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if len(w) > 2]

# document frequency: in how many chunks does each word appear?
DF = collections.Counter()
for c in CHUNKS:
    for w in set(words(c["text"])):
        DF[w] += 1

def score(query, chunk):
    """Shared words, each weighted by rarity: rare words shout, common words whisper."""
    shared = set(words(query)) & set(words(chunk["text"]))
    return sum(1.0 / DF[w] for w in shared)

def retrieve(query, k=3):
    ranked = sorted(CHUNKS, key=lambda c: -score(query, c))
    return ranked[:k]

print("baseline retriever ready")

## The question set and the baseline

In [ ]:
QUESTIONS = [
    ("How many final exam retakes do I get?", "S4.2"),
    ("What is the late work penalty?", "S4.5"),
    ("When are field trip permission forms due?", "S5.1"),
    ("When does robotics club meet?", "robotics_minutes"),
    ("What time is Wednesday dismissal?", "S2.1"),
    ("when do we get to leave early midweek", "S2.1"),
    ("Can a freshman get a varsity roster spot?", "S6.3"),
    ("How do I fight a bad grade?", "S4.3"),
    ("What time is the last bus pickup?", "bus_routes"),
    ("What does the ski trip cost?", None),
]

def run_set(retriever, k=3):
    hits = misses = 0
    detail = []
    for q, want in QUESTIONS:
        if want is None:
            detail.append((q, "n/a")); continue
        got = retriever(q, k)
        ok = any(c["section"] == want or c["doc"] == want for c in got)
        hits += ok; misses += (not ok)
        detail.append((q, "HIT" if ok else "MISS"))
    return hits, detail

base_hits, base_detail = run_set(retrieve)
print(f"baseline: {base_hits} hits of 9 answerable")
for q, r in base_detail:
    print(f"  {r:4}  {q}")

## Fix 1 — query rewriting (the phrasing mechanism)

The slang question lands in the wrong neighborhood. One cheap model call
turns it into the register the handbook uses. Offline, a lookup stands in
for that call — in Colab, `llm()` from lesson 5 does it live for any
question.

In [ ]:
REWRITES = {  # offline stand-in for: llm(f"Rewrite as a formal handbook query: {q}")
    "when do we get to leave early midweek": "what time is early dismissal on Wednesday",
    "How do I fight a bad grade?": "grade appeals department head",
}

def retrieve_rewritten(q, k=3):
    return retrieve(REWRITES.get(q, q), k)

rw_hits, rw_detail = run_set(retrieve_rewritten)
print(f"with rewriting: {rw_hits} hits (was {base_hits})")
for (q, r0), (_q, r1) in zip(base_detail, rw_detail):
    if r0 != r1:
        print(f"  {r0} -> {r1}  {q}")
assert rw_hits > base_hits, "rewriting should flip at least one miss"

## Fix 2 — raise k (the cutoff mechanism)

Anything ranked just past the cutoff is found-and-discarded. The dial has
a price: every extra chunk spends context and adds noise for the model to
ignore. Measure, don't vibe:

In [ ]:
for k in (1, 3, 5, 8):
    h, _ = run_set(retrieve_rewritten, k=k)
    chars = sum(len(c["text"]) for c in retrieve_rewritten(QUESTIONS[0][0], k))
    print(f"k={k}:  {h} hits   (~{chars} chars of context per question)")
print()
print("Past the k that captures your answers, hits flatline while cost climbs.")

## Fix 3 — the absence mechanism

The ski-trip question has no answer chunk to find. Retrieval scores stay
low across the board — which is exactly the signal ask5's refusal clause
turns into an honest 'not in the documents'. The fix is adding the
document, or letting the refusal stand.

In [ ]:
best = max(score("What does the ski trip cost?", c) for c in CHUNKS)
typical = max(score("How many final exam retakes do I get?", c) for c in CHUNKS)
print(f"best score, ski question:    {best:.2f}")
print(f"best score, retake question: {typical:.2f}")
print("A score floor (refuse to answer below it) turns absence into refusal")
print("BEFORE the model ever sees a prompt - cheaper than catching it after.")
assert best < typical/2

## Try it

1. The straddle mechanism: rebuild chunks with `overlap_sentences=0`, find
   what breaks (S6.2/S6.3 is a candidate — the rule and its petition are
   neighbors), then restore the overlap and re-measure. That's fix 4.
2. Pick the k you'd actually ship and defend it in one sentence with the
   two numbers that matter.
3. **Build turn-in:** your two worst misses diagnosed and fixed, with both
   full before/after tables and one sentence on anything a fix broke.